# 🔬 Validação Empírica do Índice Fat Tail — Tesouro Direto

## Objetivo

Responder de forma **empírica e honesta** à pergunta:

> **"O índice Fat Tail (MAD/STD) é útil para decidir QUANDO comprar títulos do Tesouro Direto?"**

A teoria por trás (Taleb, cap. 4.4.1) é elegante: quando os retornos entram em regime de cauda gorda, o mercado está em pânico — e em renda fixa de longa duração, pânico geralmente significa **taxas altas → PU baixo → oportunidade de entrada**.

Mas teoria é uma coisa; **evidência é outra**. Vamos testar.

## Metodologia

1. Baixar dados **reais** do Tesouro Direto (Tesouro Transparente CSV).
2. Selecionar títulos IPCA+ longos (os mais sensíveis ao pânico).
3. Para cada título, calcular o **iFat = MAD/STD** em janela móvel de 60 pregões.
4. Identificar **TODOS os momentos históricos** em que o iFat marcou "zona de entrada":
   - `iFat < média móvel - desvio` **E**
   - `iFat < 0.65` (limiar absoluto)
5. Para cada evento, simular **duas estratégias opostas**:
   - 🟢 **Compra**: comprar naquele dia e manter por N dias
   - 🔴 **Venda (short sintético)**: vender naquele dia e fechar em N dias
6. Medir o **retorno médio**, **hit rate**, **distribuição** e comparar com o **baseline** (retorno médio sem filtro).

Se a **compra** ganha em média, o iFat sinaliza "fundo" → bom para entrar.
Se a **venda** ganha, o iFat sinaliza que o pânico vai continuar → ruim para entrar.
Se ambas empatam, o sinal não tem valor preditivo.

In [ ]:
# Imports
import sys
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Engine do app (tem que estar no mesmo diretório)
from engine import (
    load_csv, prepare, get_series,
    compute_fat_tail_index, fat_tail_entry_signals,
    GAUSSIAN_MAD_OVER_STD,
)

plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3

print(f"Gaussian reference MAD/STD: {GAUSSIAN_MAD_OVER_STD:.4f}")

## 1. Baixar dados reais do Tesouro Direto

Fonte oficial: [Tesouro Transparente](https://www.tesourotransparente.gov.br/ckan/dataset/taxas-dos-titulos-ofertados-pelo-tesouro-direto).

In [ ]:
import os

# Tentativa 1: baixar do Tesouro Transparente
try:
    raw = load_csv()
    print(f"✅ CSV baixado online: {len(raw):,} linhas, {raw.shape[1]} colunas")
except Exception as e:
    print(f"⚠️ Não foi possível baixar online ({type(e).__name__}): {e}")
    # Tentativa 2: usar um CSV local (útil em ambientes sem acesso à internet)
    local_candidates = ["tesouro_sintetico.csv", "PrecoTaxaTesouroDireto.csv"]
    loaded = False
    for path in local_candidates:
        if os.path.exists(path):
            raw = pd.read_csv(path, sep=";", decimal=",", encoding="utf-8")
            raw.columns = raw.columns.str.strip()
            print(f"✅ Usando CSV local: {path} ({len(raw):,} linhas)")
            loaded = True
            break
    if not loaded:
        raise RuntimeError(
            "Não consegui carregar os dados. Verifique sua conexão ou baixe "
            "o CSV manualmente de https://www.tesourotransparente.gov.br e salve "
            "como 'PrecoTaxaTesouroDireto.csv' nesta pasta."
        )

print("\nColunas:", list(raw.columns))

In [ ]:
# Usamos PU Base Manhã (referência do pregão da manhã) e Taxa Compra Manhã
df = prepare(raw, taxa_col="Taxa Compra Manha", pu_col="PU Base Manha")
print(f"Dataset preparado: {len(df):,} linhas")
print(f"Período: {df['data'].min().date()} a {df['data'].max().date()}")
print(f"Títulos únicos: {df['titulo'].nunique()}")
print()
print("Amostra:")
df.head()

## 2. Selecionar títulos IPCA+ longos com bastante histórico

Queremos títulos com pelo menos 3 anos de histórico (suficiente para detectar vários regimes de pânico) e de prazo longo (porque eles são os mais voláteis, onde o iFat tem mais sinal).

In [ ]:
# Pega títulos IPCA+ (sem Juros Semestrais para simplificar) com bastante histórico
ipca_mask = df["titulo"].str.contains("IPCA", case=False, na=False)
ipca_df = df[ipca_mask].copy()

# agrupa por (titulo, vencimento) e filtra os com histórico suficiente
grouped = ipca_df.groupby(["titulo","vencimento"]).agg(
    n_obs=("data","count"),
    data_ini=("data","min"),
    data_fim=("data","max"),
).reset_index()

# pelo menos 3 anos de histórico (≈ 750 pregões)
grouped = grouped[grouped["n_obs"] >= 750]
# só vencimentos no futuro
grouped = grouped[grouped["vencimento"] > pd.Timestamp.today()]

grouped = grouped.sort_values("n_obs", ascending=False)
print(f"Títulos IPCA+ com >= 3 anos de histórico e vencimento futuro: {len(grouped)}")
grouped.head(20)

In [ ]:
# Escolhe um título IPCA+ longo com bastante histórico para análise detalhada.
# Pode trocar pelo que quiser. Vamos começar com o que tem MAIS observações.
TITULO_ALVO = grouped.iloc[0]["titulo"]
VENC_ALVO = grouped.iloc[0]["vencimento"]

print(f"Título escolhido: {TITULO_ALVO}")
print(f"Vencimento: {VENC_ALVO.date()}")
print(f"Histórico: {grouped.iloc[0]['n_obs']} pregões "
      f"({grouped.iloc[0]['data_ini'].date()} a {grouped.iloc[0]['data_fim'].date()})")

s = get_series(df, TITULO_ALVO, VENC_ALVO)
print(f"\nTamanho da série: {len(s)}")

## 3. Calcular o iFat e identificar TODAS as zonas de entrada

In [ ]:
WINDOW = 60              # janela móvel (pregões)
ABS_THRESHOLD = 0.65     # limiar absoluto do iFat (abaixo disso = cauda gorda forte)

ft = compute_fat_tail_index(s, window=WINDOW, price_col="pu", convention="mad_over_std")
ft = fat_tail_entry_signals(ft, absolute_threshold=ABS_THRESHOLD, use_moving_band=True)

print(f"Série iFat: {len(ft)} observações")
print(f"Eventos em zona de entrada: {ft['entrada'].sum()} "
      f"({ft['entrada'].mean()*100:.1f}% do tempo)")
print(f"\niFat atual: {ft['iFat'].iloc[-1]:.4f} "
      f"(ref gaussiano: {GAUSSIAN_MAD_OVER_STD:.4f})")
print(f"Status atual: {'🔴 CAUDA GORDA' if ft['entrada'].iloc[-1] else '🟢 normal'}")

In [ ]:
# Visualização: PU + iFat lado a lado
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

ax0 = axes[0]
ax0.plot(ft["data"], ft["pu"], color="#2E86AB", linewidth=1.2, label="PU")
pu_entry = ft["pu"].where(ft["entrada"], np.nan)
ax0.scatter(ft["data"], pu_entry, color="red", s=20, zorder=3, label="Zona de entrada")
ax0.set_ylabel("PU (R$)")
ax0.set_title(f"PU ao longo do tempo — {TITULO_ALVO}")
ax0.legend()

ax1 = axes[1]
ax1.plot(ft["data"], ft["iFat"], color="#6A4C93", linewidth=1.0, label="iFat")
ax1.plot(ft["data"], ft["iFat_ma"], color="#F77F00", linestyle="--", linewidth=1.0, label="média móvel")
ax1.plot(ft["data"], ft["iFat_ma_minus_sd"], color="#D62828", linestyle=":", linewidth=0.9, label="banda inferior")
ax1.axhline(GAUSSIAN_MAD_OVER_STD, color="black", alpha=0.5, label=f"Gaussiano ({GAUSSIAN_MAD_OVER_STD:.3f})")
ax1.axhline(ABS_THRESHOLD, color="gray", linestyle="--", alpha=0.7, label=f"Limiar ({ABS_THRESHOLD})")
ax1.set_ylabel("iFat")
ax1.set_xlabel("Data")
ax1.set_title("Índice Fat Tail — pontos vermelhos marcam zonas de entrada")
ax1.legend(fontsize=9)

plt.tight_layout()
plt.show()

## 4. Simular as DUAS estratégias em cada evento histórico

Para cada data em que o sinal disparou, vamos calcular o **retorno forward do PU** em 30, 60, 90, 180 e 365 dias.

- Se `ret > 0`: o PU subiu → quem **comprou** ganhou.
- Se `ret < 0`: o PU caiu → quem **vendeu** (short sintético) ganhou.

**Importante**: tratamos isso com disciplina. Ao invés de só olhar a média (que pode ser dominada por outliers), vamos olhar a distribuição completa: mediana, quartis, hit rate, etc.

In [ ]:
HORIZONS = [30, 60, 90, 180, 365]

# Para cada data da série, calcula o PU em data + h
ft_sorted = ft.sort_values("data").reset_index(drop=True)

for h in HORIZONS:
    target = ft_sorted["data"] + pd.to_timedelta(h, unit="D")
    fwd_values = []
    for td in target:
        j = ft_sorted["data"].searchsorted(td, side="left")
        fwd_values.append(ft_sorted.iloc[j]["pu"] if j < len(ft_sorted) else np.nan)
    ft_sorted[f"pu_fwd_{h}d"] = fwd_values
    ft_sorted[f"ret_{h}d"] = ft_sorted[f"pu_fwd_{h}d"] / ft_sorted["pu"] - 1

# Separa eventos vs baseline (toda a série)
eventos = ft_sorted[ft_sorted["entrada"]].copy()
baseline = ft_sorted.copy()

print(f"Eventos Fat Tail: {len(eventos)}")
print(f"Baseline (todos os dias): {len(baseline)}")

In [ ]:
# Função para calcular estatísticas de uma série de retornos
def stats_retornos(x, nome):
    x = x.dropna()
    if len(x) == 0:
        return {"estrategia": nome, "n": 0}
    return {
        "estrategia": nome,
        "n": len(x),
        "retorno_medio_%": round(x.mean() * 100, 2),
        "mediana_%": round(x.median() * 100, 2),
        "p25_%": round(x.quantile(0.25) * 100, 2),
        "p75_%": round(x.quantile(0.75) * 100, 2),
        "std_%": round(x.std() * 100, 2),
        "hit_rate_compra_%": round((x > 0).mean() * 100, 1),
        "hit_rate_venda_%": round((x < 0).mean() * 100, 1),
    }

# Para cada horizonte, compara compra vs venda nos EVENTOS vs BASELINE
all_rows = []
for h in HORIZONS:
    col = f"ret_{h}d"

    # Em evento (sinal disparou)
    ev = eventos[col].dropna()
    bs = baseline[col].dropna()

    # Compra = ficamos com o retorno "como é"
    # Venda = negamos o retorno (ganhamos se o PU cai)
    all_rows.append({**stats_retornos(ev, f"Compra no sinal ({h}d)"), "horizonte": h})
    all_rows.append({**stats_retornos(-ev, f"Venda no sinal ({h}d)"), "horizonte": h})
    all_rows.append({**stats_retornos(bs, f"Baseline ({h}d)"), "horizonte": h})

resumo = pd.DataFrame(all_rows)
resumo = resumo[["horizonte","estrategia","n","retorno_medio_%","mediana_%",
                 "p25_%","p75_%","std_%","hit_rate_compra_%","hit_rate_venda_%"]]
resumo

### Como ler essa tabela

- **Compra no sinal**: você comprou no dia do evento e vendeu N dias depois.
- **Venda no sinal**: você vendeu (short sintético) no dia do evento e recomprou N dias depois.
- **Baseline**: retorno médio de QUALQUER dia (sem filtro algum) no mesmo horizonte.

**O que queremos ver:**
- Se `retorno_medio_%` de "Compra no sinal" > Baseline → **sinal é bom para comprar** ✅
- Se `retorno_medio_%` de "Venda no sinal" > Baseline → o sinal antecede **queda** (ruim comprar)
- Se nenhum dos dois > Baseline → o sinal não ajuda a decidir direção, só mede volatilidade

**Importante sobre Venda**: shortear títulos do Tesouro Direto **não é possível na prática** para pessoa física. Essa coluna é **apenas diagnóstica** — ela nos diz se o sinal antecede queda (situação em que seria melhor **não comprar ainda**).

In [ ]:
# Visual: compara retorno médio COMPRA no sinal vs BASELINE, por horizonte
fig, ax = plt.subplots(figsize=(12, 5))

x = np.arange(len(HORIZONS))
width = 0.35

compra_means = [eventos[f"ret_{h}d"].mean() * 100 for h in HORIZONS]
baseline_means = [baseline[f"ret_{h}d"].mean() * 100 for h in HORIZONS]

bars1 = ax.bar(x - width/2, compra_means, width, label="Compra no sinal Fat Tail", color="#2E86AB")
bars2 = ax.bar(x + width/2, baseline_means, width, label="Baseline (qualquer dia)", color="gray", alpha=0.7)

ax.axhline(0, color="black", linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels([f"{h}d" for h in HORIZONS])
ax.set_ylabel("Retorno médio do PU (%)")
ax.set_title(f"Retorno após sinal Fat Tail vs baseline — {TITULO_ALVO}")
ax.legend()

# Anota valores nas barras
for bar, val in zip(bars1, compra_means):
    ax.annotate(f"{val:+.2f}%",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3 if val > 0 else -12),
                textcoords="offset points",
                ha="center", fontsize=9)
for bar, val in zip(bars2, baseline_means):
    ax.annotate(f"{val:+.2f}%",
                xy=(bar.get_x() + bar.get_width()/2, bar.get_height()),
                xytext=(0, 3 if val > 0 else -12),
                textcoords="offset points",
                ha="center", fontsize=9, alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Histogramas da distribuição dos retornos após o sinal (compra)
fig, axes = plt.subplots(1, len(HORIZONS), figsize=(5*len(HORIZONS), 4))
if len(HORIZONS) == 1:
    axes = [axes]

for ax, h in zip(axes, HORIZONS):
    vals = eventos[f"ret_{h}d"].dropna() * 100
    base_vals = baseline[f"ret_{h}d"].dropna() * 100

    ax.hist(base_vals, bins=30, alpha=0.35, color="gray", label="Baseline", density=True)
    ax.hist(vals, bins=30, alpha=0.6, color="#2E86AB", label="Após sinal", density=True)
    ax.axvline(0, color="black", linewidth=0.8)
    ax.axvline(vals.mean(), color="#2E86AB", linestyle="--", linewidth=1.5, label=f"média={vals.mean():.1f}%")
    ax.axvline(base_vals.mean(), color="gray", linestyle="--", linewidth=1.5)
    ax.set_title(f"Retorno em {h}d")
    ax.set_xlabel("Retorno (%)")
    ax.legend(fontsize=8)

plt.suptitle(f"Distribuição dos retornos: sinal vs baseline — {TITULO_ALVO}", y=1.02)
plt.tight_layout()
plt.show()

## 5. Listar os 20 eventos mais recentes

Para você VER os momentos históricos que o sinal pegou e o que aconteceu depois.

In [ ]:
# Tabela com os eventos mais recentes
cols_show = ["data", "pu", "iFat"] + [f"ret_{h}d" for h in HORIZONS]
eventos_show = eventos[cols_show].tail(20).copy()

# formatar retornos em %
for h in HORIZONS:
    eventos_show[f"ret_{h}d"] = (eventos_show[f"ret_{h}d"] * 100).round(2)

eventos_show = eventos_show.rename(columns={
    f"ret_{h}d": f"ret_{h}d (%)" for h in HORIZONS
})

print(f"Últimos 20 eventos Fat Tail em {TITULO_ALVO}:")
eventos_show

## 6. Análise agregada: MÚLTIPLOS títulos IPCA+ longos

Testar em um único título pode ser cherry-picking. Vamos rodar a mesma análise em **TODOS** os títulos IPCA+ longos com histórico suficiente e ver se a tese se sustenta no conjunto.

In [ ]:
def analisa_titulo(df, titulo, vencimento, window=60, abs_thr=0.65, horizons=(30,60,90,180,365)):
    """
    Analisa um título e retorna um dicionário com as estatísticas de compra vs baseline.
    """
    s = get_series(df, titulo, vencimento)
    if len(s) < window + max(horizons) + 30:
        return None

    try:
        ft = compute_fat_tail_index(s, window=window, price_col="pu", convention="mad_over_std")
        ft = fat_tail_entry_signals(ft, absolute_threshold=abs_thr, use_moving_band=True)
    except Exception:
        return None

    if ft.empty or ft["entrada"].sum() == 0:
        return None

    ft = ft.sort_values("data").reset_index(drop=True)
    for h in horizons:
        target = ft["data"] + pd.to_timedelta(h, unit="D")
        fwd = []
        for td in target:
            j = ft["data"].searchsorted(td, side="left")
            fwd.append(ft.iloc[j]["pu"] if j < len(ft) else np.nan)
        ft[f"ret_{h}d"] = np.array(fwd) / ft["pu"] - 1

    ev = ft[ft["entrada"]]
    bs = ft

    res = {
        "titulo": titulo,
        "vencimento": vencimento.date(),
        "n_eventos": int(ev["entrada"].sum()),
        "n_obs": len(ft),
        "pct_em_sinal": round(ev["entrada"].sum() / len(ft) * 100, 1),
    }
    for h in horizons:
        ev_ret = ev[f"ret_{h}d"].dropna()
        bs_ret = bs[f"ret_{h}d"].dropna()
        res[f"compra_medio_{h}d_%"] = round(ev_ret.mean() * 100, 2) if len(ev_ret) else None
        res[f"baseline_{h}d_%"] = round(bs_ret.mean() * 100, 2) if len(bs_ret) else None
        res[f"excesso_{h}d_%"] = (
            round((ev_ret.mean() - bs_ret.mean()) * 100, 2)
            if len(ev_ret) and len(bs_ret)
            else None
        )
        res[f"hit_compra_{h}d_%"] = round((ev_ret > 0).mean() * 100, 1) if len(ev_ret) else None
    return res


# Rode em todos os títulos IPCA+ longos (vencimento futuro, >=3 anos de histórico)
resultados = []
for _, r in grouped.iterrows():
    res = analisa_titulo(df, r["titulo"], r["vencimento"])
    if res is not None:
        resultados.append(res)

agg = pd.DataFrame(resultados)
print(f"Títulos analisados: {len(agg)}")
agg

## 7. Veredito agregado

Se a estratégia de comprar no sinal Fat Tail funciona em média, queremos ver:
- `excesso_Xd_%` positivo na maioria dos títulos e horizontes
- `hit_compra_Xd_%` acima de 50% na maioria

In [ ]:
# Resumo agregado: média dos excessos em cada horizonte
print("=" * 70)
print("VEREDITO AGREGADO — todos os títulos IPCA+ longos com histórico")
print("=" * 70)

for h in [30, 60, 90, 180, 365]:
    col = f"excesso_{h}d_%"
    hit = f"hit_compra_{h}d_%"
    if col not in agg.columns: continue
    vals = agg[col].dropna()
    hits = agg[hit].dropna()
    if len(vals) == 0: continue

    sign = "✅" if vals.mean() > 0 else "❌"
    n_pos = (vals > 0).sum()
    pct_titulos_ok = n_pos / len(vals) * 100

    print(f"\nHorizonte {h}d:")
    print(f"  {sign} Excesso médio vs baseline: {vals.mean():+.2f}%")
    print(f"     Mediana do excesso: {vals.median():+.2f}%")
    print(f"     Títulos com excesso positivo: {n_pos}/{len(vals)} ({pct_titulos_ok:.0f}%)")
    print(f"     Hit rate de compra médio: {hits.mean():.1f}%")

print()
print("=" * 70)
print("Como interpretar:")
print("  • Excesso médio > 0 E > 50% dos títulos com excesso positivo = estratégia útil")
print("  • Se os dois critérios falham: o sinal não ajuda a tomar decisão de compra")
print("  • Hit rate > 55% é excelente em qualquer estratégia de renda fixa")
print("=" * 70)

## 8. Escolhendo parâmetros: sensibilidade à janela e limiar

A resposta pode depender dos parâmetros. Vamos varrer algumas combinações e ver onde a estratégia é mais robusta.

In [ ]:
# Grid search: varia window e threshold, foca em horizonte de 90d
grid_results = []

for window in [30, 60, 90, 120]:
    for thr in [0.60, 0.65, 0.70, 0.75]:
        excessos_90d = []
        hits_90d = []
        eventos_total = 0
        for _, r in grouped.iterrows():
            res = analisa_titulo(df, r["titulo"], r["vencimento"],
                                  window=window, abs_thr=thr)
            if res is None:
                continue
            if res.get("excesso_90d_%") is not None:
                excessos_90d.append(res["excesso_90d_%"])
                hits_90d.append(res["hit_compra_90d_%"])
                eventos_total += res["n_eventos"]

        if excessos_90d:
            grid_results.append({
                "window": window,
                "threshold": thr,
                "n_titulos": len(excessos_90d),
                "eventos_total": eventos_total,
                "excesso_medio_90d_%": round(np.mean(excessos_90d), 2),
                "excesso_mediano_90d_%": round(np.median(excessos_90d), 2),
                "% títulos OK": round(np.mean(np.array(excessos_90d) > 0) * 100, 0),
                "hit_medio_90d_%": round(np.mean(hits_90d), 1),
            })

grid_df = pd.DataFrame(grid_results).sort_values("excesso_medio_90d_%", ascending=False)
print("Grid search — horizonte 90d")
print("(procura combinação de (window, threshold) com melhor excesso vs baseline)")
print()
grid_df

In [ ]:
# Heatmap visual do grid search
pivot = grid_df.pivot(index="window", columns="threshold", values="excesso_medio_90d_%")

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(pivot.values, aspect="auto", cmap="RdYlGn",
               vmin=-abs(pivot.values).max(), vmax=abs(pivot.values).max())

ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([f"{t:.2f}" for t in pivot.columns])
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
ax.set_xlabel("Threshold absoluto do iFat")
ax.set_ylabel("Janela (pregões)")
ax.set_title("Excesso médio vs baseline em 90d (%) — grid search")

# anota valores
for i in range(len(pivot.index)):
    for j in range(len(pivot.columns)):
        ax.text(j, i, f"{pivot.values[i,j]:+.2f}",
                ha="center", va="center", fontsize=10,
                color="black" if abs(pivot.values[i,j]) < pivot.values.max()*0.5 else "white")

plt.colorbar(im, ax=ax, label="Excesso (%)")
plt.tight_layout()
plt.show()

## 9. Conclusões honestas

Leia com atenção os outputs acima. As respostas que você precisa tirar:

### Pergunta 1: O sinal Fat Tail identifica momento de comprar?
Olhe para o **veredito agregado** (célula 23). Se o excesso médio é **positivo** em todos os horizontes e mais da metade dos títulos se beneficia, **sim**.

### Pergunta 2: Qual o horizonte ideal?
Normalmente o retorno se acumula em 60-180 dias — esperar 365d já costuma diluir o ganho no carrego.

### Pergunta 3: Qual o melhor (window, threshold)?
Veja o **heatmap** (célula 26). Zonas verdes grandes = configuração robusta; ponto verde isolado = provavelmente overfitting.

### 🎯 Regras de bolso para a sua decisão

1. **Só confie no sinal se o excesso se mantém em VÁRIOS horizontes** (30+60+90). Um único horizonte com excesso positivo pode ser ruído.
2. **Exija que a estratégia funcione na MAIORIA dos títulos**, não só em um. Se só o 2045 "funciona" e os outros 5 são neutros, você está vendo ruído.
3. **Hit rate é tão importante quanto o retorno médio**. Um hit rate de 45% com média de +5% vem de poucos outliers enormes — e você pode estar do lado errado quando o sinal disparar de novo.
4. **Combine com os outros sinais do app**: J4 + Z alto + iFat baixo é um stack muito mais confiável do que cada um sozinho.

### ⚠️ Limitações desta análise

- Ignoramos **custódia B3** (0,20% a.a.) no cálculo do retorno — em compras de curto prazo, isso corrói uma fração do ganho.
- Ignoramos **IR regressivo**: em resgate antes de 180d o IR é 22,5%. Sempre que o backtest medir ganho em 30/60/90 dias, o líquido é bem menor.
- Os retornos são **brutos de PU**. Para quem segura IPCA+ e resgata no vencimento, o que importa mesmo é o carrego — não o PU intermediário.
- **Shortear Tesouro Direto não é possível** para pessoa física. A coluna "Venda" é só diagnóstica.

### 🧪 Próximos experimentos que você pode fazer aqui

- Trocar `TITULO_ALVO` e rodar de novo (célula 8).
- Mudar `WINDOW` e `ABS_THRESHOLD` (célula 10).
- Adicionar filtros de regime (ex.: só disparar se a taxa Selic estiver acima da média histórica).
- Combinar Fat Tail com o sinal J4/Z do Scanner (requires stacking de duas condições).

In [ ]:
agg_sorted = agg.sort_values("excesso_90d_%", ascending=False)
agg_sorted[["titulo", "vencimento", "n_eventos", 
            "excesso_30d_%", "excesso_90d_%", "excesso_365d_%",
            "hit_compra_90d_%"]]